In [1]:
import json
import re
import sqlite3

In [2]:
from openai import OpenAI
import instructor
from pydantic import BaseModel

In [3]:
topic = "Linux"

In [4]:
def slugify_title(title):
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")


def get_connection(db_path="../quiz_outlines.db"):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


In [5]:
def get_outline(conn, article_title):
    row = conn.execute(
        "SELECT article_id, outline_json FROM outlines WHERE article_title = ?",
        (article_title,),
    ).fetchone()
    if row is None:
        return None
    return {
        "article_id": row["article_id"],
        "outline": json.loads(row["outline_json"])
    }


In [6]:
conn = get_connection()
outline = get_outline(conn, topic)

In [7]:
outline

{'article_id': 'linux',
 'outline': {'article_title': 'Linux',
  'sections': [{'breadcrumb': 'Introduction',
    'preview': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17...',
    'token_count': 369},
   {'breadcrumb': 'Overview',
    'preview': 'The Linux kernel was created by Linus Torvalds, following the lack of a working kernel for GNU, a Unix-compatible operating system made entirely of...',
    'token_count': 590},
   {'breadcrumb': 'History > Precursors',
    'preview': "The Unix operating system was conceived of and implemented in 1969, at AT&T's Bell Labs in the United States, by Ken Thompson, Dennis Ritchie,...",
    'token_count': 537},
   {'breadcrumb': 'History > Creation',
    'preview': 'While attending the University of Helsinki in the fall of 1990, Torvalds enrolled in a Unix course. The course used a MicroVAX minicomputer running...',
    'token_count': 442},
   {'br

In [8]:
from pydantic import BaseModel

class QuizOutline(BaseModel):
    section_breadcrumb: str
    question_count: int
    difficulty: str
    reason: str

In [9]:
PLANNER_SYSTEM_PROMPT = """
You are an assessment planner.

Your task is to design a blueprint for a quiz using only the article outline that you are given.

The outline contains:
- hierarchical section names (breadcrumbs)
- a short preview of each section
- the approximate size of each section

Do NOT generate quiz questions.
Do NOT retrieve information.
Do NOT invent facts that are not implied by the outline.

Your job is only to decide:

1. Which sections should contribute questions.
2. How many questions should come from each section.
3. What difficulty each section should contribute.
4. Why each section was selected.

When creating the blueprint:

- Prefer broad coverage over concentrating questions in a single section.
- Ensure every selected section appears relevant to the user's request.
- Avoid selecting sections that appear too small or too narrow unless they are specifically relevant.
- Large sections may receive multiple questions.
- Introductory sections should usually receive fewer questions than substantive sections.
- If the user requests an easier quiz, favor foundational sections.
- If the user requests a harder quiz, favor advanced or specialized sections.
- The total number of planned questions MUST equal the requested number.

Return ONLY valid JSON.
"""

In [10]:
PLANNER_PROMPT = f"""
User request:

Generate a medium-difficulty quiz about Linux.

Number of questions:
10

Article outline:

{outline}
"""

In [11]:
from google import genai
from google.genai import types
import os

In [12]:
or_client = instructor.from_provider(
    "deepseek/deepseek-v4-flash",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    async_client=False,
)

In [11]:
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [13]:
resp = or_client.create(
    messages=[
        {
            "role": "system",
            "content": PLANNER_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": PLANNER_PROMPT
        }
    ],
    response_model=list[QuizOutline],
    extra_body={"provider": {"require_parameters": True}},
)


In [14]:
resp

[QuizOutline(section_breadcrumb='Overview', question_count=2, difficulty='medium', reason='Provides a broad foundation of what Linux is and its kernel origins, suitable for medium difficulty introductory questions.'),
 QuizOutline(section_breadcrumb='History > Creation', question_count=1, difficulty='medium', reason='Covers the origin story of Linux by Linus Torvalds, a key historical topic that tests understanding of how Linux began.'),
 QuizOutline(section_breadcrumb='History > Copyright, trademark, and naming', question_count=1, difficulty='medium', reason="Covers licensing (GPL) and naming, important for understanding Linux's open-source legal framework."),
 QuizOutline(section_breadcrumb='Design', question_count=2, difficulty='medium', reason='A large, substantive section covering the architectural philosophy and design choices of Linux, good for medium-difficulty questions.'),
 QuizOutline(section_breadcrumb='Development', question_count=1, difficulty='medium', reason='Covers the

In [17]:
response = gemini_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=PLANNER_PROMPT,
    config=types.GenerateContentConfig(system_instruction=PLANNER_SYSTEM_PROMPT, 
    response_mime_type="application/json",
    response_schema=list[QuizOutline])
)

In [19]:
response.parsed

[QuizOutline(section_breadcrumb='Introduction', question_count=1, difficulty='Medium', reason='Provides basic foundational concepts of the Linux OS family, establishing essential context for a medium-difficulty quiz.'),
 QuizOutline(section_breadcrumb='Overview', question_count=1, difficulty='Medium', reason="Covers the origin of the Linux kernel and its connection to GNU, key to understanding Linux's overall structure."),
 QuizOutline(section_breadcrumb='History > Precursors', question_count=1, difficulty='Medium', reason="Explores the Unix roots which directly influenced Linux's design philosophy."),
 QuizOutline(section_breadcrumb='History > Copyright, trademark, and naming', question_count=1, difficulty='Medium', reason="Discusses the GNU GPL license and licensing constraints, a significant aspect of Linux's history and legal framework."),
 QuizOutline(section_breadcrumb='Usage > Market share and uptake', question_count=1, difficulty='Medium', reason='A large section covering quant

In [15]:
response = resp

In [16]:
# blueprint = [item.model_dump(exclude={"reason"}) for item in response.parsed]
blueprint = [item.model_dump(exclude={"reason"}) for item in response]
blueprint

[{'section_breadcrumb': 'Overview',
  'question_count': 2,
  'difficulty': 'medium'},
 {'section_breadcrumb': 'History > Creation',
  'question_count': 1,
  'difficulty': 'medium'},
 {'section_breadcrumb': 'History > Copyright, trademark, and naming',
  'question_count': 1,
  'difficulty': 'medium'},
 {'section_breadcrumb': 'Design', 'question_count': 2, 'difficulty': 'medium'},
 {'section_breadcrumb': 'Development',
  'question_count': 1,
  'difficulty': 'medium'},
 {'section_breadcrumb': 'Development > Community',
  'question_count': 1,
  'difficulty': 'medium'},
 {'section_breadcrumb': 'Development > Programming on Linux',
  'question_count': 1,
  'difficulty': 'medium'},
 {'section_breadcrumb': 'Usage > Market share and uptake',
  'question_count': 1,
  'difficulty': 'medium'}]

In [17]:
def batch_blueprint(blueprint, items_per_batch=3):
    return [
        blueprint[i : i + items_per_batch]
        for i in range(0, len(blueprint), items_per_batch)
    ]


In [18]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")


/home/sanjeeb/Projects/quiz/backend/.venv/lib/python3.13/site-packages/qdrant_client/qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [22]:
for item in blueprint:
    retrieved_context = client.scroll(
        collection_name="Quiz-App-Dev-Collection",
        scroll_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="section_breadcrumb",
                    match=models.MatchValue(value=item["section_breadcrumb"]),
                ),
            ]
        ),
    )
    # print(retrieved_context[0][0].payload)
    item["article_title"] = retrieved_context[0][0].payload["article_title"]
    item["text"] = retrieved_context[0][0].payload["raw_text"]
    item["source_url"] = retrieved_context[0][0].payload["source_url"]



In [67]:
blueprint

[{'section_breadcrumb': 'Introduction',
  'question_count': 1,
  'difficulty': 'Easy',
  'reasoning': 'Provides an accessible introductory question to establish the definition of Linux and its Unix-like open-source nature.',
  'article_title': 'Linux',
  'text': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17 September 1991 by Linus Torvalds. Some members of the family are typically packaged as a distribution (a.k.a. distro), which includes the kernel alongside supporting system software and libraries developed by third parties—such as GNU, Red Hat, and X.Org—to create a complete operating system; however, not all Linux-based operating systems are considered distros, with Android being an example. Linux was originally designed as a clone of Unix and is distributed under the copyleft GPL license.\nThere are many thousands of Linux distributions, many based directly or indirectly on othe

In [23]:
def build_batch_prompt(sections_with_meta):
    blocks = []
    for i, item in enumerate(sections_with_meta, start=1):
        blocks.append(f"""Section {i}:
Article: {item["article_title"]}
Breadcrumb: {item["section_breadcrumb"]}
Difficulty: {item["difficulty"]}
Number of questions required: {item["question_count"]}

Passage:
\"\"\"
{item["text"]}
\"\"\"""")

    joined = "\n\n---\n\n".join(blocks)
    return f"""Generate quiz questions for the following sections.

Each section is independent.

For every section:

- Use ONLY that section's reference material.
- Generate exactly the requested number of questions.
- Respect the requested difficulty.
- Each generated question must include the correct section_index.

Sections


{joined}

"""


In [24]:
batched_blueprint = batch_blueprint(blueprint)
batched_blueprint

[[{'section_breadcrumb': 'Overview',
   'question_count': 2,
   'difficulty': 'medium',
   'article_title': 'Linux',
   'text': "The Linux kernel was created by Linus Torvalds, following the lack of a working kernel for GNU, a Unix-compatible operating system made entirely of free software that had been in development since 1983 by the GNU Project, led by Richard Stallman. A working Unix system called Minix was later released but its license was not entirely free at the time and it was made for education purposes. The first entirely free Unix for personal computers, 386BSD, did not appear until 1992, by which time Torvalds had already built and publicly released the first version of the Linux kernel on the Internet. Like GNU and 386BSD, Linux did not have any Unix code, being a fresh re-implementation, and therefore avoided legal issues from AT&T. Linux distributions became popular in the 1990s and made Unix technologies accessible to home users on personal computers whereas previously

In [25]:
QUESTION_GENERATOR_SYSTEM_PROMPT = """
You are an expert instructional designer and assessment writer.

Your task is to generate high-quality multiple-choice questions from reference material.

The reference material is divided into independent sections. Treat each section independently. Never combine information from different sections, even if they belong to the same article.

Before writing any questions for a section, internally determine the most important concepts presented in that section. Prefer assessing concepts over isolated facts. Do NOT reveal this reasoning.

Question Selection Priority (highest to lowest):

1. Core concepts and definitions
2. Relationships between concepts
3. Purpose, significance, or design rationale
4. Cause-and-effect relationships
5. Historical developments that explain why something exists
6. Important terminology
7. Examples used to illustrate a concept
8. Minor exceptions or edge cases
9. Trivia

When multiple questions are requested for the same section:

- Each question must assess a DIFFERENT important concept.
- Avoid asking two questions that test the same underlying knowledge.
- Maximize coverage of the section.

Difficulty Guidelines

Easy
- Tests one important concept or definition.
- The answer is explicitly stated.
- Should require understanding, not merely locating a word.

Medium
- Tests understanding of relationships, comparisons, motivations, consequences, or historical context.
- May require connecting multiple RELATED ideas from the same section.
- Never combine unrelated facts simply to increase difficulty.

Hard
- Tests subtle distinctions, reasoning, or interpretation using information from the section.
- The answer must still be fully supported by the reference material.
- Do not require outside knowledge.

Question Writing Guidelines

A good question:

- sounds like it belongs on a university quiz
- focuses on understanding rather than memorization
- is concise and unambiguous
- has exactly one clearly correct answer
- avoids unnecessary wording
- can be answered entirely from the provided section

Avoid:

- "According to the passage..."
- quoting large portions of the reference
- testing obscure facts when more important concepts exist
- asking about the same idea twice
- questions whose answer is obvious because one option is much longer or more specific than the others

Options

- Exactly four options.
- Exactly one correct answer.
- Distractors should be plausible for someone with partial understanding.
- Distractors should represent common misconceptions when possible.
- Keep options similar in length and style.
- Avoid "All of the above" and "None of the above."

Explanations

Provide a brief explanation explaining why the correct answer is correct.

Return ONLY valid JSON matching the provided schema."""

In [27]:
class QuestionsResponse(BaseModel):
    section_index: int
    question: str
    options: list[str]
    correct_answer: str
    explanation: str

In [31]:
import time

In [33]:
questions = []

for blueprint in batched_blueprint:
    start_time = time.perf_counter()
    resp = or_client.create(
        messages=[
            {
                "role": "system",
                "content": QUESTION_GENERATOR_SYSTEM_PROMPT,
            },
            {"role": "user", "content": build_batch_prompt(blueprint)},
        ],
        response_model=list[QuestionsResponse],
        extra_body={"provider": {"require_parameters": True}},
    )
    questions.extend(resp)
    end_time = time.perf_counter()
    print(f"Elapsed time: {end_time - start_time:.4f} seconds")
    


Elapsed time: 117.1629 seconds
Elapsed time: 22.2208 seconds
Elapsed time: 11.0598 seconds


In [34]:
questions

[QuestionsResponse(section_index=0, question='What primary problem motivated Linus Torvalds to create the Linux kernel?', options=['The lack of a fully free Unix-compatible operating system kernel', 'The high cost of proprietary Unix licenses for personal computers', 'The poor performance of the Minix operating system on desktop hardware', 'A desire to create an operating system compatible with Microsoft Windows'], correct_answer='The lack of a fully free Unix-compatible operating system kernel', explanation="The passage states the Linux kernel was created by Linus Torvalds 'following the lack of a working kernel for GNU' and that Minix was not fully free, and 386BSD did not appear until 1992, by which time Torvalds had already released Linux. The motivating problem was the absence of a free Unix-compatible kernel."),
 QuestionsResponse(section_index=0, question='What is the significance of the system call exception in the GPLv2 license of the Linux kernel?', options=['It requires all 

In [26]:
questions = ""
for blueprint in batched_blueprint:
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash",
        config=types.GenerateContentConfig(system_instruction=QUESTION_GENERATOR_SYSTEM_PROMPT),
        contents=build_batch_prompt(blueprint),
    )
    questions += response.text
    print("first batch of questions generated...")


NameError: name 'gemini_client' is not defined

In [ ]:
question_filepath = "questions.txt"

if os.path.exists(question_filepath):
    os.remove(question_filepath)

with open(question_filepath, "a") as f:
    for question in questions:
        f.write(f"{question.question}\n")
        for i, option in enumerate(question.options):
            f.write(f"{i+1}. {option}\n")

        f.write("\n")
